# Experiment E2 -- Cross Entropy + Data Augmentation (RQ-1)

**Thesis:** Segmentasi Zona Anatomi pada Citra MRI Pasien Kanker Prostat
menggunakan 2D U-Net dan Rekonstruksi 3D.

**RQ-1:** *Berapa Hasil Evaluasi dari Segmentasi Citra MRI pada Pasien Kanker
Prostat menggunakan 2D U-Net?* -- E2 is supporting evidence under RQ-1.

**Research question (E2):** *Does adding mild, anatomically-plausible
training-time augmentation improve validation segmentation performance while
keeping all other major training conditions -- including the baseline Cross
Entropy loss -- unchanged?*

The outcome is **not presumed**. E2 may improve, match, or regress against the
baseline; the measured validation result decides.

---

## 1. Experiment overview

### Position in the 2x2 design

| | **No augmentation** | **Augmentation** |
|---|---|---|
| **CE loss** | Baseline *(done)* | **E2 <- this notebook** |
| **Dice + CE loss** | E1 *(done)* | E1+E2 *(later, not here)* |

E2 keeps the **baseline Cross Entropy loss** on purpose, so augmentation is
isolated from the E1 loss factor. E2 trains from a **fresh initialization** --
it is never warm-started from the Baseline or E1 checkpoint.

### Controlled comparison

| | Baseline | E1 | **E2** |
|---|---|---|---|
| Loss | Cross Entropy | 1.0 CE + 1.0 Soft Dice | **Cross Entropy** (= baseline) |
| Augmentation | none | none | **ON (training slices only)** |
| Architecture | 2D U-Net (1 -> 3) | identical | identical |
| Optimizer / LR / WD | AdamW / 1e-3 / 1e-4 | identical | identical |
| Batch size / epochs / seed | 8 / 100 / 42 | identical | identical |
| Precision | FP32 (AMP off) | identical | identical |
| Scheduler | none | none | none |
| Preprocessing | per-volume norm, crop/pad 442 | identical | identical |
| Validation | 20 official cases | identical | identical |

**The single experimental factor changed from Baseline to E2 is AUGMENTATION = ON.**

### Reference validation results (to be reported alongside E2)

| Arm | CG Dice | PZ Dice | Macro Dice |
|---|---|---|---|
| Baseline | 0.8642 | 0.7375 | 0.8008 |
| E1 | 0.8662 | 0.7527 | 0.8094 |
| **E2** | *this run* | *this run* | *this run* |

**Selection vs comparison** (two different metrics, never conflated):

* *within-run checkpoint selection* -- the batch-level foreground-macro validation
  Dice proxy in `src/train.py::validate`, **unchanged** from Baseline and E1;
* *across-experiment comparison* -- per-case, volume-level, original-voxel-space
  **macro Dice = (CG + PZ) / 2** over the 20 official validation cases.

### Scope limits enforced by this notebook

* The official **19-case test set is NOT used** anywhere.
* No external validation (PROSTATEx / MSD) is touched.
* **No final model is selected** and **no 2x2 factorial analysis is run** --
  both wait until the E1+E2 arm exists.

---

## How to run

1. `Runtime > Change runtime type > T4 GPU`
2. `Runtime > Run all`
3. Approve the Google Drive mount prompt when it appears.

That is the only manual interaction required.

### If Colab disconnects or the T4 session times out

Reconnect the runtime and **Run all** again. With the default `RESUME = "auto"`
the notebook detects `latest_checkpoint.pt` on Drive, restores the model,
optimizer, RNG and sampler state, reloads the complete epoch history, and
continues from the next epoch. **No previously recorded epoch is ever lost.**


## 2. Reproducibility configuration

`RESUME` is the only switch you may need to touch, and normally you do not:

| Value | Behaviour |
|---|---|
| `"auto"` **(default)** | Resume if a usable checkpoint exists, start fresh if not, skip training if already at 100 epochs, and stop with a clear error if the checkpoint is corrupt. Correct for both first run and every re-run. |
| `True` | Require a checkpoint. Fail loudly if there is none (protects against silently restarting a long run from scratch). |
| `False` | Require a clean directory. Fail loudly if a checkpoint or history already exists (protects an existing run from being overwritten). |

Everything else below is fixed by the committed config and must not be edited here.


In [ ]:
# ============================== THE ONLY SWITCH ==============================
RESUME = 'auto'      # 'auto' (recommended) | True (must resume) | False (must be fresh)
# =============================================================================

assert RESUME in ('auto', True, False), "RESUME must be 'auto', True, or False"

# Locations. These follow the existing project convention used by the
# baseline / E1 notebooks: <DRIVE_ROOT>/results/<experiment name>/.
DRIVE_ROOT          = '/content/drive/MyDrive/THESIS_PROSTATE158'
REPO_URL            = 'https://github.com/amadeussandro/prostate-mri-segmentation-thesis.git'
REPO_DIR            = '/content/thesis-project'
REPO_BRANCH         = 'main'
EXPERIMENT_NAME     = 'exp_e2_ce_aug'
E2_CONFIG_RELPATH   = 'configs/experiments/exp_e2_ce_aug.yaml'

# Invariants asserted against the committed config later in this notebook.
# They are NOT applied to the config -- they are checked against it, so a
# silent drift in either place stops the run.
EXPECTED = {
    'architecture'  : 'ProstateUNet2D',
    'in_channels'   : 1,
    'out_channels'  : 3,
    'init_features' : 32,
    'dropout_rate'  : 0.2,
    'loss'          : 'cross_entropy',
    'optimizer'     : 'AdamW',
    'learning_rate' : 0.001,
    'weight_decay'  : 0.0001,
    'batch_size'    : 8,
    'num_epochs'    : 100,
    'seed'          : 42,
    'amp'           : False,
    'num_workers'   : 0,
    'target_mask'   : 't2_anatomy_reader1.nii.gz',
    'normalization' : 'per_volume',
    'z_resample'    : False,
    'spatial_mode'  : 'crop_pad',
    'target_size'   : [442, 442],
    'augmentation'  : True,
}

EXPECTED_AUGMENTATION = {
    'rotation_degrees': 10.0, 'rotation_p': 0.5,
    'scale_min': 0.9, 'scale_max': 1.1, 'scale_p': 0.3,
    'hflip_p': 0.5,
    'gamma_min': 0.7, 'gamma_max': 1.5, 'gamma_p': 0.3,
    'seed': 42,
}

# Published anchors for the comparison table (validation, 20 cases).
BASELINE_REFERENCE = {'cg': 0.8642, 'pz': 0.7375, 'macro': 0.8008}
E1_REFERENCE       = {'cg': 0.8662, 'pz': 0.7527, 'macro': 0.8094}

print(f'RESUME mode       : {RESUME!r}')
print(f'Experiment        : {EXPERIMENT_NAME}')
print(f'Drive root        : {DRIVE_ROOT}')
print(f'Config (committed): {E2_CONFIG_RELPATH}')


In [ ]:
# Global seeding happens inside src.train.run_training (src/utils.set_seed).
# Seeding here as well makes the notebook's own verification cells (section 9,
# section 13) reproducible too; it cannot affect training, which re-seeds.
import os, random
SEED = EXPECTED['seed']
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
print(f'Notebook-level seed: {SEED}')


## 3. Colab environment setup / GPU verification

Hard-fails rather than silently training a 100-epoch run on CPU.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'])


In [ ]:
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None
DEVICE = 'cuda' if CUDA_AVAILABLE else 'cpu'

print(f'CUDA available: {CUDA_AVAILABLE}')
print(f'GPU: {GPU_NAME}')
print(f'Device being used: {DEVICE}')
print(f'torch: {torch.__version__}')

if not CUDA_AVAILABLE:
    raise RuntimeError(
        'No CUDA GPU detected. A full 100-epoch E2 run must NOT fall back to CPU.\n'
        'Fix: Runtime > Change runtime type > T4 GPU, then Runtime > Run all.'
    )
if GPU_NAME and 'T4' not in GPU_NAME:
    print(f'\nNOTE: expected a T4; got {GPU_NAME}. Training will still run, but'
          ' epoch timings will not be comparable to the T4 reference.')


## 4. Mount Google Drive

**This is the core of the resume safety.** Every piece of permanent experiment
state -- checkpoints, per-epoch metrics, history, run state, evaluation, report --
is written to Drive, never to Colab's ephemeral `/content`. When the VM is
recycled, nothing is lost.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 5. Repository / project path detection

Clones the repo on a fresh VM, pulls on a re-run, and records the exact commit
the results were produced from.


In [ ]:
import os, subprocess, sys

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'checkout', REPO_BRANCH], check=True)
subprocess.run(['git', 'pull', 'origin', REPO_BRANCH], check=True)

GIT_COMMIT = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                            capture_output=True, text=True).stdout.strip()
print('Repo      :', REPO_DIR)
print('Branch    :', REPO_BRANCH)
print('HEAD      :', GIT_COMMIT)
subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'], check=True)
print('\nWorking tree (should be clean in a fresh clone):')
subprocess.run(['git', '-C', REPO_DIR, 'status', '--short'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

E2_CONFIG_REPO = os.path.join(REPO_DIR, E2_CONFIG_RELPATH)
assert os.path.isfile(E2_CONFIG_REPO), f'Committed E2 config not found: {E2_CONFIG_REPO}'
print('\nE2 config :', E2_CONFIG_REPO)


## 6. Dependency installation


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nibabel', 'scipy', 'pandas', 'pyyaml', 'matplotlib'], check=True)
import nibabel, scipy, pandas, yaml, matplotlib
print('nibabel', nibabel.__version__, '| scipy', scipy.__version__,
      '| pandas', pandas.__version__, '| matplotlib', matplotlib.__version__)


## 7. Dataset path verification

The Prostate158 train+validation archive is read from Drive and cached on the
local Colab disk (much faster random access during training), then symlinked
into the repo so the committed relative dataset paths resolve unchanged.


In [ ]:
import os, shutil, subprocess

DRIVE_DATASET_DIR = os.path.join(DRIVE_ROOT, 'dataset')
DRIVE_RESULTS_DIR = os.path.join(DRIVE_ROOT, 'results')
LOCAL_DATASET_DIR = '/content/local_dataset'
USE_LOCAL_DATASET_CACHE = True   # set False to read NIfTI directly from Drive

assert os.path.isdir(DRIVE_DATASET_DIR), f'Dataset not found on Drive: {DRIVE_DATASET_DIR}'
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

if USE_LOCAL_DATASET_CACHE:
    os.makedirs(LOCAL_DATASET_DIR, exist_ok=True)
    print('Syncing dataset Drive -> local (safe to re-run)...')
    rc = subprocess.run(['rsync', '-a', DRIVE_DATASET_DIR.rstrip('/') + '/',
                         LOCAL_DATASET_DIR.rstrip('/') + '/']).returncode
    if rc != 0:
        shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR, dirs_exist_ok=True)
    DATASET_SRC = LOCAL_DATASET_DIR
else:
    DATASET_SRC = DRIVE_DATASET_DIR

link_path = os.path.join(REPO_DIR, 'dataset')
if os.path.islink(link_path):
    os.remove(link_path)
elif os.path.exists(link_path):
    raise RuntimeError(f'{link_path} exists and is not a symlink -- refusing to overwrite.')
os.symlink(DATASET_SRC, link_path)
print('Linked', link_path, '->', os.readlink(link_path))

DS_ROOT   = os.path.join(REPO_DIR, 'dataset', 'prostate158_train', 'train')
TRAIN_CSV = os.path.join(REPO_DIR, 'dataset', 'prostate158_train', 'train.csv')
VALID_CSV = os.path.join(REPO_DIR, 'dataset', 'prostate158_train', 'valid.csv')
for p in (DS_ROOT, TRAIN_CSV, VALID_CSV):
    assert os.path.exists(p), f'Required dataset path missing: {p}'
print('\nDataset root :', DS_ROOT)
print('Train CSV    :', TRAIN_CSV)
print('Valid CSV    :', VALID_CSV)


## 8. Dataset integrity checks

Cache completeness, the official 119/20 split, and label validity. The official
19-case test archive must be **absent**, and the notebook asserts that.


In [ ]:
from src.data_integrity import format_cache_report, verify_dataset_cache

report = verify_dataset_cache(DS_ROOT, TRAIN_CSV, VALID_CSV)
print(format_cache_report(report))
assert bool(report['ok']), 'Dataset cache INCOMPLETE -- fix before training (see report above).'
print('\nDataset integrity: OK')


In [ ]:
import numpy as np, nibabel as nib
from src.splits import load_official_split

split = load_official_split(train_csv=TRAIN_CSV, valid_csv=VALID_CSV)
print(f'Official split -> train {len(split.train_ids)} | val {len(split.val_ids)} |',
      f'test {len(split.test_ids) if split.test_ids else 0} (test intentionally absent)')
assert len(split.train_ids) == 119, f'expected 119 training cases, got {len(split.train_ids)}'
assert len(split.val_ids) == 20, f'expected 20 validation cases, got {len(split.val_ids)}'
assert not (set(split.train_ids) & set(split.val_ids)), 'train/val overlap detected'
assert split.test_ids is None, 'the official test split must NOT be available to E2'

print('\nValidation cases (the SAME 20 used by Baseline and E1):')
print(' ', ', '.join(split.val_ids))

seen = set()
for pid in (split.train_ids[:3] + split.val_ids[:3]):
    m = nib.load(os.path.join(DS_ROOT, pid, 't2_anatomy_reader1.nii.gz')).get_fdata()
    seen |= set(int(round(v)) for v in np.unique(m))
    assert os.path.exists(os.path.join(DS_ROOT, pid, 't2.nii.gz')), f'missing t2 for {pid}'
print('\nObserved mask labels (sampled cases):', sorted(seen))
assert seen.issubset({0, 1, 2}), f'Unexpected labels: {seen}'
print('Label check: OK (0=background, 1=CG, 2=PZ)')


### 8b. Scientific-control tests

Mechanically asserts that E2 differs from the baseline **only** by augmentation,
that the loss is still plain Cross Entropy, that augmentation is train-only and
label-safe, that history survives a resume, and that E2 cannot resume from a
Baseline or E1 checkpoint. If any of this has drifted, the run stops here.


In [ ]:
import subprocess, sys
rc = subprocess.run([sys.executable, '-m', 'pytest', '-q',
                     'tests/test_e2_config.py',
                     'tests/test_augment.py',
                     'tests/test_history_preservation.py',
                     'tests/test_losses.py'],
                    cwd=REPO_DIR).returncode
assert rc == 0, 'Scientific-control tests FAILED -- do not train until resolved.'
print('\nScientific control: VERIFIED')


## 9. Augmentation verification

E2's independent variable, checked before a single epoch is spent.

The transform set is **not invented here**. It is the pre-committed E2
specification from `Outputs/rq1_experiment_specification.md` section 6,
implemented in `src/augment.py` and declared in the committed config:

| Transform | Range | Probability | Applies to |
|---|---|---|---|
| random rotation | +/- 10 deg | 0.5 | image (bilinear) + mask (nearest) |
| random scale / zoom | 0.9 - 1.1 | 0.3 | image (bilinear) + mask (nearest) |
| horizontal flip | -- | 0.5 | image + mask |
| random gamma | 0.7 - 1.5 | 0.3 | **image only** |

Deliberately excluded (would distort zonal anatomy): vertical flip, heavy
elastic deformation, large rotations.

Ordering: rotation and scale are composed into **one** affine and applied with a
**single** interpolation per array (so the image is not blurred twice and thin PZ
structures are not eroded twice), then the flip, then gamma.

Where it is applied: `src/train.py::_build_dataset(..., augment=True)` for the
**training** dataset only. The validation dataset is built with `augment=False`.


In [ ]:
from src.augment import AugmentationConfig, build_train_augmentor
from src.utils import load_config

committed = load_config(E2_CONFIG_REPO)
aug_cfg = AugmentationConfig.from_config(committed)

print('Resolved augmentation spec')
print('-' * 60)
for k, v in aug_cfg.to_dict().items():
    print(f'  {k:<28}: {v}')
print('-' * 60)
print(aug_cfg.describe())

assert aug_cfg.enabled, 'E2 requires augmentation to be ENABLED.'
for key, expected in EXPECTED_AUGMENTATION.items():
    actual = getattr(aug_cfg, key)
    assert abs(float(actual) - float(expected)) < 1e-9, \
        f'augmentation.{key} = {actual}, expected {expected} (pre-committed spec)'
assert aug_cfg.to_dict()['mask_interpolation_order'] == 0, 'mask must use nearest-neighbour'
assert aug_cfg.to_dict()['image_interpolation_order'] == 1, 'image must use bilinear'
print('\nAugmentation spec matches the pre-committed E2 specification.')


In [ ]:
# Behavioural verification on a REAL validation-cohort-independent training slice.
import numpy as np
from src.dataset import ProstateZonal2DDataset
from src.transforms import PreprocessingConfig

probe_ds = ProstateZonal2DDataset(
    dataset_root=DS_ROOT, patient_ids=[split.train_ids[0]],
    image_name='t2.nii.gz', mask_name=committed['data']['target_mask'],
    slice_sampling='all',
    preprocessing_config=PreprocessingConfig.from_dict(committed['preprocessing']),
    cache_data=True)

# Pick a slice that actually contains both zones, so label preservation is testable.
probe_index = next((i for i in range(len(probe_ds))
                    if set(np.unique(probe_ds[i]['mask']).tolist()) >= {0, 1, 2}),
                   len(probe_ds) // 2)
reference = probe_ds[probe_index]
print(f'Probe: case {reference["patient_id"]} slice {reference["slice_idx"]} '
      f'(index {probe_index} of {len(probe_ds)})')
print('  image shape', reference['image'].shape, reference['image'].dtype)
print('  mask  shape', reference['mask'].shape, reference['mask'].dtype,
      'labels', sorted(set(np.unique(reference['mask']).tolist())))

augmentor = build_train_augmentor(committed)
assert augmentor is not None, 'build_train_augmentor returned None for the E2 config'

aug_checks = []
def acheck(name, ok, detail=''):
    aug_checks.append((name, bool(ok), detail))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}{(' -- ' + str(detail)) if detail else ''}")

outputs = [augmentor(reference) for _ in range(200)]

acheck('A1. image shape preserved',
       all(o['image'].shape == reference['image'].shape for o in outputs))
acheck('A2. mask shape preserved',
       all(o['mask'].shape == reference['mask'].shape for o in outputs))
acheck('A3. dtypes preserved',
       all(o['image'].dtype == np.float32 and o['mask'].dtype == np.int64 for o in outputs))
observed_labels = set()
for o in outputs:
    observed_labels |= set(int(v) for v in np.unique(o['mask']))
acheck('A4. labels remain a subset of {0,1,2}',
       observed_labels.issubset({0, 1, 2}), sorted(observed_labels))
acheck('A5. all pixel values finite',
       all(np.isfinite(o['image']).all() for o in outputs))
acheck('A6. augmentation actually changes samples',
       any(not np.array_equal(o['image'], reference['image']) for o in outputs))
acheck('A7. input sample is not mutated in place',
       set(int(v) for v in np.unique(reference['mask'])).issubset({0, 1, 2}))
acheck('A8. case/slice identity carried through',
       all(o['patient_id'] == reference['patient_id'] and
           o['slice_idx'] == reference['slice_idx'] for o in outputs))


In [ ]:
# Empirical firing rates must match the configured probabilities.
counts = augmentor.counts
n = counts['calls']
print(f'Draws observed: {n}')
for key, expected in (('rotation', aug_cfg.rotation_p), ('scale', aug_cfg.scale_p),
                      ('hflip', aug_cfg.hflip_p), ('gamma', aug_cfg.gamma_p)):
    rate = counts[key] / n
    ok = abs(rate - expected) < 0.12   # generous band: n=200 is a small sample
    acheck(f'A9.{key} fires at ~p={expected}', ok, f'observed {rate:.3f}')

# Determinism: the same seed must replay the same sequence.
a = build_train_augmentor(committed)
b = build_train_augmentor(committed)
same = all(np.array_equal(a(reference)['image'], b(reference)['image']) for _ in range(20))
acheck('A10. seed 42 replays an identical augmentation sequence', same)


In [ ]:
# The decisive wiring check: TRAIN is augmented, VALIDATION is not.
from src.train import _build_dataset
from src.augment import SliceAugmentor

probe_cfg = {k: (dict(v) if isinstance(v, dict) else v) for k, v in committed.items()}
probe_cfg['data'] = dict(committed['data'])
probe_cfg['data']['dataset_root'] = DS_ROOT

train_probe = _build_dataset(probe_cfg, [split.train_ids[0]], augment=True)
val_probe   = _build_dataset(probe_cfg, [split.val_ids[0]],   augment=False)

acheck('A11. TRAINING dataset carries the augmentor',
       isinstance(train_probe.transform, SliceAugmentor))
acheck('A12. VALIDATION dataset is NEVER augmented',
       val_probe.transform is None)
acheck('A13. validation samples are deterministic',
       np.array_equal(val_probe[0]['image'], val_probe[0]['image']) and
       np.array_equal(val_probe[0]['mask'], val_probe[0]['mask']))
acheck('A14. num_workers is 0 (augmentation RNG must not be forked)',
       int(committed['training'].get('num_workers', 0)) == 0)
acheck('A15. no test data referenced',
       'test_csv' not in committed['data'] and 'test_dir' not in committed['data'])

aug_failed = [name for name, ok, _ in aug_checks if not ok]
print(f'\nAugmentation verification: {len(aug_checks) - len(aug_failed)}/{len(aug_checks)} passed')
assert not aug_failed, f'Augmentation verification FAILED: {aug_failed}'
print('Augmentation is correct, label-safe, reproducible, and TRAIN-ONLY.')


In [ ]:
# Visual confirmation: original vs three augmented draws, image and mask together.
import matplotlib.pyplot as plt

viz = build_train_augmentor(committed)
draws = [reference] + [viz(reference) for _ in range(3)]
titles = ['ORIGINAL (no augmentation)', 'augmented #1', 'augmented #2', 'augmented #3']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, (sample, title) in enumerate(zip(draws, titles)):
    img = sample['image'][0]
    msk = sample['mask']
    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(title, fontsize=10)
    axes[0, col].axis('off')
    axes[1, col].imshow(img, cmap='gray')
    axes[1, col].imshow(np.ma.masked_where(msk == 0, msk), cmap='autumn', alpha=0.55,
                        vmin=0, vmax=2)
    axes[1, col].set_title('image + mask overlay', fontsize=9)
    axes[1, col].axis('off')
fig.suptitle('E2 augmentation -- image and mask receive the IDENTICAL spatial transform',
             fontsize=12)
fig.tight_layout()
plt.show()

del probe_ds, train_probe, val_probe, viz, a, b, outputs


## 10. Model definition

Built by `src/model.py::build_model` from the committed config -- the notebook
defines no architecture of its own. Asserted identical to Baseline / E1.


In [ ]:
import torch
from src.model import build_model

model_cfg = committed['model']
for key in ('architecture', 'in_channels', 'out_channels', 'init_features', 'dropout_rate'):
    assert model_cfg[key] == EXPECTED[key], \
        f'model.{key} = {model_cfg[key]!r}, expected {EXPECTED[key]!r} (must match Baseline/E1)'

model = build_model(committed).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Architecture : {model_cfg['architecture']}")
print(f"Input/output : {model_cfg['in_channels']} -> {model_cfg['out_channels']} channels")
print(f"Init features: {model_cfg['init_features']} | dropout {model_cfg['dropout_rate']}")
print(f'Parameters   : {n_params:,}')

dummy = torch.zeros(1, 1, 442, 442, device=DEVICE)
model.eval()
with torch.no_grad():
    logits = model(dummy)
assert tuple(logits.shape) == (1, 3, 442, 442), f'unexpected output shape {tuple(logits.shape)}'
assert torch.isfinite(logits).all()
print(f'Forward pass : OK, logits {tuple(logits.shape)}')
print('\nModel is identical to the Baseline/E1 architecture (E2 changes nothing here).')


## 11. Loss definition

**E2 uses the BASELINE loss.** Plain Cross Entropy -- not E1's Dice+CE. Inheriting
E1's loss here would confound the two factors and make the 2x2 design unreadable,
so the notebook asserts it.


In [ ]:
from src.losses import CROSS_ENTROPY, LossConfig, compute_loss_components
from torch import nn

loss_config = LossConfig.from_config(committed)
print('Loss spec    :', loss_config.to_dict())
print('Description  :', loss_config.describe())
assert loss_config.name == CROSS_ENTROPY, \
    f'E2 must use plain Cross Entropy, got {loss_config.name!r}'
assert loss_config.is_baseline_cross_entropy
assert 'loss' not in committed, 'E2 must not declare a top-level `loss:` block'

# Prove it is bit-identical to nn.CrossEntropyLoss, i.e. the baseline objective.
torch.manual_seed(SEED)
probe_logits = torch.randn(2, 3, 32, 32, device=DEVICE)
probe_target = torch.randint(0, 3, (2, 32, 32), device=DEVICE)
ours, components = compute_loss_components(probe_logits, probe_target, loss_config)
reference_loss = nn.CrossEntropyLoss()(probe_logits, probe_target.long())
assert torch.allclose(ours, reference_loss, atol=0, rtol=0), \
    'E2 loss is NOT bit-identical to nn.CrossEntropyLoss'
print(f'\nCE loss check: ours {ours.item():.8f} == nn.CrossEntropyLoss {reference_loss.item():.8f}')
print('Dice term    :', components['dice'], '(0.0 -- not part of the E2 objective)')
print('\nLoss is UNCHANGED from the baseline. Augmentation is the only E2 factor.')


## 12. Optimizer

AdamW, lr 1e-3, weight decay 1e-4 -- identical to Baseline and E1. Constructed
inside `run_training`; verified here against the committed config and exercised
with one real step on a real slice.


In [ ]:
training_cfg = committed['training']
for key in ('optimizer', 'learning_rate', 'weight_decay', 'batch_size',
            'num_epochs', 'seed', 'num_workers'):
    assert training_cfg.get(key) == EXPECTED[key], \
        f'training.{key} = {training_cfg.get(key)!r}, expected {EXPECTED[key]!r}'
assert bool(training_cfg.get('amp', False)) is False, 'AMP must be OFF (FP32) for E2'
assert training_cfg.get('loss_function') == 'cross_entropy'

optimizer = torch.optim.AdamW(model.parameters(),
                              lr=training_cfg['learning_rate'],
                              weight_decay=training_cfg['weight_decay'])
print(f"Optimizer    : {training_cfg['optimizer']}")
print(f"Learning rate: {training_cfg['learning_rate']}")
print(f"Weight decay : {training_cfg['weight_decay']}")
print(f"Scheduler    : NONE (constant LR)")
print(f"Precision    : FP32 (AMP off)")

# One real optimizer step on one real augmented slice.
model.train()
aug_sample = build_train_augmentor(committed)(reference)
images = torch.from_numpy(aug_sample['image']).unsqueeze(0).float().to(DEVICE)
targets = torch.from_numpy(aug_sample['mask']).unsqueeze(0).long().to(DEVICE)
before = model.final_conv.weight.detach().clone()
optimizer.zero_grad(set_to_none=True)
step_logits = model(images)
step_loss, _ = compute_loss_components(step_logits, targets, loss_config)
step_loss.backward()
assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None), \
    'non-finite gradient on an augmented sample'
optimizer.step()
assert not torch.equal(before, model.final_conv.weight.detach()), 'optimizer step had no effect'
print(f'\nBackward + step on an AUGMENTED sample: OK (loss {step_loss.item():.6f})')

del model, optimizer, step_logits, step_loss, images, targets, before, dummy, logits
torch.cuda.empty_cache()


## 13. DataLoader

`run_training` builds the real loaders. This cell verifies the wiring on a small
subset (a few cases) rather than materializing all 119 training volumes twice --
the full loaders are built once, inside training.

Training uses the seeded `WeightedRandomSampler` (foreground/background balancing,
unchanged from Baseline / E1); validation uses the complete, unweighted, unshuffled
slice population.


In [ ]:
from torch.utils.data import DataLoader
from src.train import _build_train_sampler, _dataloader_perf_kwargs

subset_train_ids = split.train_ids[:2]
subset_val_ids   = split.val_ids[:1]

train_subset = _build_dataset(probe_cfg, subset_train_ids, augment=True)
val_subset   = _build_dataset(probe_cfg, subset_val_ids,   augment=False)
print(f'Subset train slices: {len(train_subset)} from {subset_train_ids}')
print(f'Subset val   slices: {len(val_subset)} from {subset_val_ids}')

sampler = _build_train_sampler(train_subset, EXPECTED['seed'],
                               training_cfg.get('use_weighted_sampling', True))
perf = _dataloader_perf_kwargs(training_cfg, DEVICE)
print('DataLoader perf kwargs:', perf)
assert perf['num_workers'] == 0, 'augmented training requires num_workers=0'

train_loader = DataLoader(train_subset, batch_size=training_cfg['batch_size'],
                          shuffle=(sampler is None), sampler=sampler, **perf)
val_loader   = DataLoader(val_subset, batch_size=training_cfg['batch_size'],
                          shuffle=False, **perf)

batch = next(iter(train_loader))
print('\nTrain batch  : image', tuple(batch['image'].shape), batch['image'].dtype,
      '| mask', tuple(batch['mask'].shape), batch['mask'].dtype)
assert tuple(batch['image'].shape) == (training_cfg['batch_size'], 1, 442, 442)
assert tuple(batch['mask'].shape) == (training_cfg['batch_size'], 442, 442)
assert set(batch['mask'].unique().tolist()).issubset({0, 1, 2})
assert torch.isfinite(batch['image']).all()

vbatch = next(iter(val_loader))
print('Val batch    : image', tuple(vbatch['image'].shape), '| mask', tuple(vbatch['mask'].shape))
assert set(vbatch['mask'].unique().tolist()).issubset({0, 1, 2})

print('\nDataLoader wiring: OK (train augmented + weighted-sampled, val plain).')
del train_subset, val_subset, train_loader, val_loader, batch, vbatch, sampler


## 14. Training configuration

### E2 experiment contract

The committed config `configs/experiments/exp_e2_ce_aug.yaml` is the source of
truth. The **only** field overridden for Colab is `experiment.output_dir`, pointed
at Drive so everything survives a disconnect. The cell asserts nothing else differs,
and asserts every invariant in the contract below.

| Field | Required value |
|---|---|
| Model | same 2D U-Net as Baseline/E1, 1 -> 3 channels, 32 init features, dropout 0.2 |
| Loss | Cross Entropy **only** |
| Optimizer | AdamW |
| Learning rate | 1e-3 |
| Weight decay | 1e-4 |
| Batch size | 8 |
| Maximum epochs | 100 |
| Seed | 42 |
| Precision | FP32 |
| AMP | OFF |
| Scheduler | NONE |
| Preprocessing | exactly as Baseline/E1 (per-volume norm, no z-resample, crop/pad 442x442) |
| Dataset | Prostate158 |
| Target | `t2_anatomy_reader1` |
| Validation | the same 20 official validation cases as Baseline/E1 |
| Test | **must not be accessed** |
| External validation | **must not be accessed** |
| **Augmentation** | **ON** -- the single experimental factor |


In [ ]:
import copy, yaml

E2_DRIVE_DIR       = os.path.join(DRIVE_RESULTS_DIR, EXPERIMENT_NAME)
BASELINE_DRIVE_DIR = os.path.join(DRIVE_RESULTS_DIR, 'exp01_baseline')
E1_DRIVE_DIR       = os.path.join(DRIVE_RESULTS_DIR, 'exp_e1_dice_ce')
os.makedirs(E2_DRIVE_DIR, exist_ok=True)

cfg = copy.deepcopy(committed)
cfg['experiment']['output_dir'] = E2_DRIVE_DIR

# Guard: output_dir is the ONLY permitted deviation from the committed config.
changed = sorted(k for k in set(committed) | set(cfg) if committed.get(k) != cfg.get(k))
assert changed == ['experiment'], f'Unexpected config deviation: {changed}'
for k in set(committed['experiment']) | set(cfg['experiment']):
    if k != 'output_dir':
        assert committed['experiment'].get(k) == cfg['experiment'].get(k), k

# ----- E2 experiment contract, asserted field by field -----
contract = []
def require(label, actual, expected):
    ok = actual == expected
    contract.append((label, ok, f'{actual!r}'))
    print(f"[{'OK  ' if ok else 'FAIL'}] {label:<34}: {actual!r}"
          f"{'' if ok else f'   <-- EXPECTED {expected!r}'}")

require('model.architecture',   cfg['model']['architecture'],   EXPECTED['architecture'])
require('model.in_channels',    cfg['model']['in_channels'],    EXPECTED['in_channels'])
require('model.out_channels',   cfg['model']['out_channels'],   EXPECTED['out_channels'])
require('model.init_features',  cfg['model']['init_features'],  EXPECTED['init_features'])
require('model.dropout_rate',   cfg['model']['dropout_rate'],   EXPECTED['dropout_rate'])
require('loss',                 LossConfig.from_config(cfg).name, EXPECTED['loss'])
require('training.optimizer',   cfg['training']['optimizer'],   EXPECTED['optimizer'])
require('training.learning_rate', cfg['training']['learning_rate'], EXPECTED['learning_rate'])
require('training.weight_decay', cfg['training']['weight_decay'], EXPECTED['weight_decay'])
require('training.batch_size',  cfg['training']['batch_size'],  EXPECTED['batch_size'])
require('training.num_epochs',  cfg['training']['num_epochs'],  EXPECTED['num_epochs'])
require('training.seed',        cfg['training']['seed'],        EXPECTED['seed'])
require('training.amp (FP32)',  bool(cfg['training'].get('amp', False)), EXPECTED['amp'])
require('training.num_workers', cfg['training'].get('num_workers', 0), EXPECTED['num_workers'])
require('scheduler',            'lr_scheduler' in cfg['training'], False)
require('data.target_mask',     cfg['data']['target_mask'],     EXPECTED['target_mask'])
require('preproc.normalization', cfg['preprocessing']['normalization'], EXPECTED['normalization'])
require('preproc.z_resample',   cfg['preprocessing']['z_resample'], EXPECTED['z_resample'])
require('preproc.spatial_mode', cfg['preprocessing']['spatial_mode'], EXPECTED['spatial_mode'])
require('preproc.target_size',  list(cfg['preprocessing']['target_size']), EXPECTED['target_size'])
require('augmentation.enabled', AugmentationConfig.from_config(cfg).enabled, EXPECTED['augmentation'])
require('test split referenced', bool(cfg['data'].get('test_csv') or cfg['data'].get('test_dir')), False)

violations = [label for label, ok, _ in contract if not ok]
assert not violations, f'E2 CONTRACT VIOLATED: {violations}'
print(f'\nE2 contract: {len(contract)}/{len(contract)} invariants satisfied.')

E2_CONFIG_COLAB = '/content/config_e2_ce_aug_colab.yaml'
with open(E2_CONFIG_COLAB, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, default_flow_style=False)

NUM_EPOCHS   = cfg['training']['num_epochs']
LATEST_CKPT  = os.path.join(E2_DRIVE_DIR, 'latest_checkpoint.pt')
BEST_CKPT    = os.path.join(E2_DRIVE_DIR, 'best_model.pt')
METRICS_CSV  = os.path.join(E2_DRIVE_DIR, 'metrics.csv')
HISTORY_JSON = os.path.join(E2_DRIVE_DIR, 'training_history.json')
STATE_JSON   = os.path.join(E2_DRIVE_DIR, 'state', 'training_state.json')

print('\nE2 config (Colab):', E2_CONFIG_COLAB)
print('Drive output dir :', E2_DRIVE_DIR)
print('Max epochs       :', NUM_EPOCHS)


### 14b. Configuration fingerprint

Printed here and persisted to Drive alongside the results by `run_training`, so
the exact settings a result came from are always recoverable.


In [ ]:
from src.utils import build_config_fingerprint, format_config_fingerprint

fingerprint = build_config_fingerprint(cfg, device=DEVICE)
print(format_config_fingerprint(fingerprint))

assert fingerprint['amp'] is False, 'AMP must be OFF for E2 (FP32 baseline comparison).'
assert fingerprint['precision'] == 'FP32'
assert fingerprint['seed'] == 42
assert fingerprint['num_epochs'] == 100
assert fingerprint['augmentation_enabled'] is True
assert fingerprint['loss'].lower().startswith('cross'), fingerprint['loss']
assert fingerprint['references_test_split'] is False
print('\nAMP: OFF | Precision: FP32 | Augmentation: ON | Test split: NOT referenced')


## 15. Checkpoint / resume system

### What is persisted to Drive, and when

| Artifact | Written | Contents |
|---|---|---|
| `latest_checkpoint.pt` | **after every epoch**, atomically | model, optimizer, epoch, best metric, best epoch, full config, python/numpy/torch RNG state, sampler generator state |
| `best_model.pt` | whenever the validation proxy improves | model, optimizer, epoch, best metric, best epoch, config |
| `metrics.csv` | after every epoch, whole-file atomic rewrite | one row per epoch |
| `training_history.json` | after every epoch, atomically | richer per-epoch record + augmentation and loss spec |
| `state/training_state.json` | after every epoch, atomically | last completed epoch, best epoch/metric, epochs remaining, resume flag |
| `config.yaml`, `config_fingerprint.{json,txt}` | at run start | the resolved configuration |

Every write goes through a temp file + `os.replace`, so a session killed
mid-write can never leave a truncated checkpoint or a half-written history.

### History is never overwritten

`run_training` reloads the existing history from Drive **before** the epoch loop
and merges new epochs into it keyed by epoch number
(`src/train.py::load_existing_history` / `upsert_epoch_record`). Resuming at epoch
57 therefore ends with epochs **1..100** on disk, not 57..100. Re-running an epoch
that already exists updates that one row in place instead of appending a duplicate.

### Configuration compatibility

`verify_checkpoint_compatibility` refuses to resume when the checkpoint's embedded
config disagrees with the current one on experiment name, architecture, loss,
**augmentation**, preprocessing, dataset split, optimizer, LR, weight decay, batch
size, seed or AMP. That is what makes it impossible to continue E2 from the
Baseline or E1 checkpoint, or to silently flip augmentation off half-way through.

### Epoch budget

Maximum 100 epochs. A checkpoint at epoch 63 resumes at 64; a checkpoint at 100 is
left alone and training is skipped. The **best** epoch is whichever epoch won the
validation proxy -- epoch 100 is never assumed to be the best model.


In [ ]:
import json
from src.train import inspect_checkpoint, load_existing_history, plan_training_run

info = inspect_checkpoint(LATEST_CKPT)
print('latest_checkpoint.pt ->')
for k in ('exists', 'readable', 'epoch', 'best_epoch', 'best_metric', 'resumable',
          'has_model_state', 'has_optimizer_state', 'has_rng_state', 'size_bytes', 'error'):
    print(f'  {k:<22}: {info.get(k)}')

existing_history = load_existing_history(HISTORY_JSON, METRICS_CSV)
recorded_epochs = [int(r['epoch']) for r in existing_history]

plan = plan_training_run(LATEST_CKPT, NUM_EPOCHS)
print()
print(plan['message'])

if plan['action'] == 'corrupt':
    raise RuntimeError('Existing E2 checkpoint is unusable -- see message above. '
                       'It has NOT been modified or deleted; investigate before re-running.')

# --- Reconcile the plan with the user's RESUME setting.
if RESUME is True and plan['action'] == 'fresh':
    raise RuntimeError(
        f'RESUME=True but no usable checkpoint exists at {LATEST_CKPT}.\n'
        'Refusing to silently restart a 100-epoch run from scratch. '
        'Set RESUME="auto" (or False) if a fresh run really is what you want.'
    )
if RESUME is False and plan['action'] in ('resume', 'already_complete'):
    raise RuntimeError(
        f"RESUME=False but an existing run was found at {E2_DRIVE_DIR} "
        f"(checkpoint epoch {plan['checkpoint_epoch']}, "
        f'{len(existing_history)} epoch record(s) on disk).\n'
        'Refusing to overwrite it. Set RESUME="auto" to continue that run, or move '
        'the directory aside first if you genuinely want to start over.'
    )

RESUME_FROM = plan['resume_from']
SKIP_TRAINING = plan['action'] == 'already_complete'

print()
print('=' * 78)
print('RESUME STATUS')
print('=' * 78)
print(f"  Checkpoint found        : {'YES -- ' + LATEST_CKPT if info['exists'] else 'NO'}")
print(f"  Action                  : {plan['action'].upper()}")
print(f"  Resuming from epoch     : {plan['start_epoch'] if plan['action'] == 'resume' else '-'}")
_best_epoch = plan.get('best_epoch')
_best_metric = plan.get('best_metric')
print(f"  Best epoch so far       : {_best_epoch if _best_epoch is not None else 'n/a'}")
_best_metric_text = f'{_best_metric:.6f}' if isinstance(_best_metric, float) else 'n/a'
print(f'  Best validation metric  : {_best_metric_text}')
print(f'  History length on disk  : {len(existing_history)} epoch record(s)'
      + (f' (epochs {min(recorded_epochs)}..{max(recorded_epochs)})' if recorded_epochs else ''))
print(f'  Epoch budget            : {NUM_EPOCHS}')
print(f'  Output directory        : {E2_DRIVE_DIR}')
print('=' * 78)

if os.path.isfile(STATE_JSON):
    with open(STATE_JSON) as f:
        print('\nPrevious run state:', json.dumps(json.load(f), indent=2)[:1200])


In [ ]:
# Explicit fresh-initialization guard: E2 must never start from another arm.
from src.train import verify_checkpoint_compatibility

for other_name, other_dir in (('baseline', BASELINE_DRIVE_DIR), ('E1', E1_DRIVE_DIR)):
    foreign = os.path.join(other_dir, 'best_model.pt')
    assert os.path.abspath(foreign) != os.path.abspath(RESUME_FROM or ''), \
        f'E2 must NOT resume from the {other_name} checkpoint'

if RESUME_FROM:
    ckpt_cfg = torch.load(RESUME_FROM, map_location='cpu', weights_only=False).get('config')
    result = verify_checkpoint_compatibility(ckpt_cfg, cfg, RESUME_FROM)
    if result['verified']:
        print('Checkpoint compatibility: VERIFIED -- same experiment, model, loss,')
        print('  augmentation, preprocessing, split and hyperparameters.')
    else:
        print('Checkpoint compatibility: SKIPPED --', result['skipped_reason'])
    assert (ckpt_cfg or {}).get('experiment', {}).get('name') in (None, EXPERIMENT_NAME), \
        'the resume checkpoint belongs to a different experiment'
else:
    print('FRESH INITIALIZATION: E2 starts from randomly initialized weights.')
    print('  It is NOT warm-started from the Baseline or E1 checkpoint.')


## 16. Training (max 100 epochs, FP32, augmented, auto-resume)

All state goes straight to Drive after every epoch. If this cell is interrupted,
reconnect the runtime and **Run all** again -- it continues at the next epoch and
the epoch history already on Drive is reloaded and appended to, never replaced.


In [ ]:
import time
from src.train import run_training

print('Experiment   :', EXPERIMENT_NAME)
print('Loss         : Cross Entropy (baseline)')
print('Augmentation : ON (training slices only)')
print('AMP          : OFF')
print('Precision    : FP32')
print(f'Device       : {DEVICE} ({GPU_NAME})')
print(f'Resume from  : {RESUME_FROM or "(fresh run)"}')
print()

if SKIP_TRAINING:
    print(f'E2 already completed {NUM_EPOCHS} epochs -- NOT retraining.')
    train_summary = None
else:
    t0 = time.time()
    train_summary = run_training(
        E2_CONFIG_COLAB,
        resume_from=RESUME_FROM,
        history_path=HISTORY_JSON,
    )
    print(f'\nElapsed: {(time.time() - t0) / 3600.0:.2f} h')
    print()
    for k, v in train_summary.items():
        print(f'  {k:<20}: {v}')


In [ ]:
# History integrity check, immediately after training.
final_history = load_existing_history(HISTORY_JSON, METRICS_CSV)
final_epochs = [int(r['epoch']) for r in final_history]
print(f'Epoch records on disk: {len(final_history)}')
if final_epochs:
    print(f'Epoch range          : {min(final_epochs)}..{max(final_epochs)}')
    gaps = sorted(set(range(1, max(final_epochs) + 1)) - set(final_epochs))
    duplicates = len(final_epochs) != len(set(final_epochs))
    assert not duplicates, 'duplicate epoch rows found in the history'
    if gaps:
        print(f'WARNING: missing epoch record(s): {gaps}')
        print('  Training itself was continuous; these rows were never written (or were')
        print('  lost before the history-preservation fix) and cannot be reconstructed.')
    else:
        print('History is COMPLETE and contiguous -- no epoch was lost across interruptions.')
if recorded_epochs:
    preserved = set(recorded_epochs).issubset(set(final_epochs))
    print(f'Pre-resume epochs preserved: {preserved} '
          f'({len(recorded_epochs)} before -> {len(final_epochs)} now)')
    assert preserved, 'HISTORY LOSS DETECTED: epochs recorded before this session are gone.'


## 17. Validation

The per-epoch validation signal that ran inside training: the batch-level
foreground-macro Dice proxy, **unchanged** from Baseline and E1. This is the
*selection* metric, not the comparison metric.

The validation set was never augmented.


In [ ]:
import pandas as pd

history_df = pd.read_csv(METRICS_CSV)
print(f'Per-epoch records: {len(history_df)}')
print('\nFirst 5 epochs:')
print(history_df.head(5).to_string(index=False))
print('\nLast 5 epochs:')
print(history_df.tail(5).to_string(index=False))

print('\nValidation proxy summary:')
print(f"  best   : {history_df['val_dice_proxy'].max():.6f} "
      f"(epoch {int(history_df.loc[history_df['val_dice_proxy'].idxmax(), 'epoch'])})")
print(f"  final  : {history_df['val_dice_proxy'].iloc[-1]:.6f} "
      f"(epoch {int(history_df['epoch'].iloc[-1])})")
print(f"  mean CG (slice-level): {history_df['val_dice_class1_CG'].tail(10).mean():.6f} (last 10 epochs)")
print(f"  mean PZ (slice-level): {history_df['val_dice_class2_PZ'].tail(10).mean():.6f} (last 10 epochs)")


## 18. Best checkpoint selection

The evaluated model is the epoch that won the within-run validation proxy --
**not** automatically epoch 100. If the best epoch is 81, epoch 81 is the model.


In [ ]:
best_info = inspect_checkpoint(BEST_CKPT)
latest_info = inspect_checkpoint(LATEST_CKPT)

assert best_info['exists'] and best_info['readable'], f'No usable best_model.pt at {BEST_CKPT}'

proxy_best_epoch = int(history_df.loc[history_df['val_dice_proxy'].idxmax(), 'epoch'])
proxy_best_value = float(history_df['val_dice_proxy'].max())

print('=' * 78)
print('BEST CHECKPOINT SELECTION')
print('=' * 78)
print(f"  Selection rule          : max batch-level foreground-macro val Dice proxy")
print(f"                            (identical rule to Baseline and E1)")
print(f"  Best epoch (checkpoint) : {best_info['epoch']}")
print(f"  Best epoch (history)    : {proxy_best_epoch}")
print(f"  Best proxy value        : {proxy_best_value:.6f}")
print(f"  Last epoch trained      : {latest_info['epoch']}")
print(f"  Best == last epoch?     : {best_info['epoch'] == latest_info['epoch']}")
print(f"  best_model.pt           : {BEST_CKPT}")
print(f"  latest_checkpoint.pt    : {LATEST_CKPT}")
print('=' * 78)

if best_info['epoch'] != proxy_best_epoch:
    print('\nNOTE: the checkpoint epoch and the history argmax differ. This is expected',
          'only if an epoch record was lost; otherwise investigate before reporting.')
if best_info['epoch'] != latest_info['epoch']:
    print(f"\nThe best model is epoch {best_info['epoch']}, NOT the final epoch "
          f"{latest_info['epoch']}. That is a valid and expected outcome.")


## 19. Per-case evaluation (20 official validation cases)

Per-case, volume-level, original-voxel-space metrics -- Dice, IoU, HD95, ASD,
precision, recall -- for the selected checkpoint. Inference only, FP32,
no augmentation.

**The official 19-case test set is not touched.**


In [ ]:
from src.evaluate import run_evaluation

eval_result = run_evaluation(
    config_path=E2_CONFIG_COLAB,
    checkpoint_path=BEST_CKPT,
    output_dir=E2_DRIVE_DIR,
    split='val',            # NEVER 'test' in E2
    class_ids=(0, 1, 2),
)
print('\nPer-case metrics written to:',
      os.path.join(E2_DRIVE_DIR, 'eval_val_per_case_metrics.csv'))


In [ ]:
per_case = pd.read_csv(os.path.join(E2_DRIVE_DIR, 'eval_val_per_case_metrics.csv'))
print(f'Per-case rows: {len(per_case)}')
print('\nColumns:', list(per_case.columns))
print()
print(per_case.to_string(index=False))


## 20. Aggregate metrics

Primary: CG Dice, PZ Dice, macro Dice. Secondary: IoU, HD95, ASD, precision, recall.


In [ ]:
summary = eval_result['summary']
CG, PZ = 1, 2

e2_cg    = summary[CG]['dice']['mean']
e2_pz    = summary[PZ]['dice']['mean']
e2_macro = (e2_cg + e2_pz) / 2

print('=' * 78)
print('E2 AGGREGATE VALIDATION METRICS (20 cases, per-case, original voxel space)')
print('=' * 78)
print(f'  PRIMARY')
print(f'    CG Dice    : {e2_cg:.4f}')
print(f'    PZ Dice    : {e2_pz:.4f}')
print(f'    MACRO Dice : {e2_macro:.4f}')
print('-' * 78)
print(f'  SECONDARY')
for class_id, label in ((CG, 'CG'), (PZ, 'PZ')):
    for metric in ('iou', 'hd95_mm', 'asd_mm', 'precision', 'recall'):
        cell = summary[class_id].get(metric)
        if cell is not None:
            sd = cell.get('std')
            sd_text = f" (SD {sd:.4f})" if isinstance(sd, float) else ''
            print(f'    {label} {metric:<10}: {cell["mean"]:.4f}{sd_text}')
print('=' * 78)


## 21. Comparison with Baseline / E1

E2 is reported **alongside** the baseline and E1. No final model is selected here,
and the 2x2 factorial analysis is **not** run -- that waits until E1+E2 exists.

The baseline's and E1's `eval_val_summary.json` are read from Drive (they are
generated artefacts and are intentionally not committed to Git).


In [ ]:
import subprocess, sys

baseline_summary_path = os.path.join(BASELINE_DRIVE_DIR, 'eval_val_summary.json')
e1_summary_path       = os.path.join(E1_DRIVE_DIR, 'eval_val_summary.json')

if not os.path.isfile(baseline_summary_path):
    raise FileNotFoundError(
        f'Baseline validation summary not found on Drive: {baseline_summary_path}\n'
        'It is a generated artefact (gitignored). Re-run the baseline validation '
        'evaluation, or copy the existing file to that path, then re-run this cell.'
    )
if not os.path.isfile(e1_summary_path):
    print(f'NOTE: E1 summary not found at {e1_summary_path}; the report will')
    print('      compare E2 against the baseline only.')

report_cmd = [sys.executable, 'scripts/finalize_e2_report.py',
              '--e2-dir', E2_DRIVE_DIR,
              '--baseline-dir', BASELINE_DRIVE_DIR]
if os.path.isfile(e1_summary_path):
    report_cmd += ['--e1-dir', E1_DRIVE_DIR]
rc = subprocess.run(report_cmd, cwd=REPO_DIR).returncode
assert rc == 0, 'E2 report generation failed.'


In [ ]:
print('=' * 82)
print('VALIDATION MACRO DICE -- 2x2 DESIGN PROGRESS (20 official validation cases)')
print('=' * 82)
print(f"{'Arm':<26}{'CG Dice':>10}{'PZ Dice':>10}{'Macro':>10}{'vs baseline':>14}")
print('-' * 82)
print(f"{'Baseline (CE, no aug)':<26}{BASELINE_REFERENCE['cg']:>10.4f}"
      f"{BASELINE_REFERENCE['pz']:>10.4f}{BASELINE_REFERENCE['macro']:>10.4f}{'--':>14}")
print(f"{'E1 (Dice+CE, no aug)':<26}{E1_REFERENCE['cg']:>10.4f}"
      f"{E1_REFERENCE['pz']:>10.4f}{E1_REFERENCE['macro']:>10.4f}"
      f"{E1_REFERENCE['macro'] - BASELINE_REFERENCE['macro']:>+14.4f}")
print(f"{'E2 (CE, aug)  <- THIS RUN':<26}{e2_cg:>10.4f}{e2_pz:>10.4f}{e2_macro:>10.4f}"
      f"{e2_macro - BASELINE_REFERENCE['macro']:>+14.4f}")
print(f"{'E1+E2 (Dice+CE, aug)':<26}{'not run':>10}{'not run':>10}{'not run':>10}{'--':>14}")
print('-' * 82)
print('  E2 - E1 macro Dice: '
      f'{e2_macro - E1_REFERENCE["macro"]:+.4f}  (descriptive only -- E1 and E2 vary')
print('    DIFFERENT factors, so this is not a ranking and not a decision rule)')
print('=' * 82)
print()
print('NOT concluded here:')
print('  * no final model is selected;')
print('  * the 2x2 factorial analysis is NOT run (it needs the E1+E2 cell);')
print('  * the official test set and external validation remain untouched.')


In [ ]:
from IPython.display import Image, Markdown, display

for name in ('training_loss.png', 'validation_dice.png', 'validation_metrics.png'):
    p = os.path.join(E2_DRIVE_DIR, 'plots', name)
    if os.path.isfile(p):
        display(Image(filename=p))

report_path = os.path.join(E2_DRIVE_DIR, 'training_report.md')
if os.path.isfile(report_path):
    display(Markdown(open(report_path, encoding='utf-8').read()))


## 22. Artifact verification

Confirms every required E2 deliverable exists on Drive and is non-empty, so a
later session (or your local VS Code, after syncing Drive) finds a complete run.


In [ ]:
REQUIRED = [
    ('best checkpoint',        os.path.join(E2_DRIVE_DIR, 'best_model.pt')),
    ('latest checkpoint',      os.path.join(E2_DRIVE_DIR, 'latest_checkpoint.pt')),
    ('per-epoch metrics',      METRICS_CSV),
    ('training history',       HISTORY_JSON),
    ('training state',         STATE_JSON),
    ('experiment config',      os.path.join(E2_DRIVE_DIR, 'config.yaml')),
    ('config fingerprint',     os.path.join(E2_DRIVE_DIR, 'config_fingerprint.json')),
    ('per-case metrics',       os.path.join(E2_DRIVE_DIR, 'eval_val_per_case_metrics.csv')),
    ('aggregate summary CSV',  os.path.join(E2_DRIVE_DIR, 'eval_val_summary.csv')),
    ('aggregate summary JSON', os.path.join(E2_DRIVE_DIR, 'eval_val_summary.json')),
    ('experiment summary',     os.path.join(E2_DRIVE_DIR, 'summary.json')),
    ('training report',        os.path.join(E2_DRIVE_DIR, 'training_report.md')),
    ('plot: training loss',    os.path.join(E2_DRIVE_DIR, 'plots', 'training_loss.png')),
    ('plot: validation dice',  os.path.join(E2_DRIVE_DIR, 'plots', 'validation_dice.png')),
    ('plot: validation metrics', os.path.join(E2_DRIVE_DIR, 'plots', 'validation_metrics.png')),
]

missing = []
print(f"{'Artifact':<26}{'Size':>12}  Path")
print('-' * 100)
for label, path in REQUIRED:
    exists = os.path.isfile(path)
    size = os.path.getsize(path) if exists else 0
    if not exists or size == 0:
        missing.append(label)
    print(f"{label:<26}{(f'{size:,}' if exists else 'MISSING'):>12}  {path}")
print('-' * 100)

# Forbidden artifacts: nothing produced from the test split or external
# validation. Matched on the actual artifact naming (`eval_test_*` /
# `*_test.*`), NOT a bare 'test' substring -- that would flag
# `laTESTt_checkpoint.pt`.
import re as _re
_forbidden_pattern = _re.compile(r'(^|[^a-z])(eval_test|test_split|prostatex|msd)',
                                 _re.IGNORECASE)
forbidden = [f for f in os.listdir(E2_DRIVE_DIR) if _forbidden_pattern.search(f)]
print('Test / external-validation artifacts in the E2 output dir:',
      forbidden or 'NONE (correct)')
assert not forbidden, f'E2 output directory contains forbidden artifacts: {forbidden}'

assert not missing, f'Missing or empty E2 artifacts: {missing}'
print('\nAll required E2 artifacts are present on Drive.')


In [ ]:
# Reproducibility record, written next to the results.
import json, platform, sys, time

repro = {
    'experiment': EXPERIMENT_NAME,
    'generated_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'git_commit': GIT_COMMIT,
    'git_branch': REPO_BRANCH,
    'repo_url': REPO_URL,
    'notebook': 'notebooks/prostate158_exp_e2_ce_aug_colab.ipynb',
    'config_committed': E2_CONFIG_RELPATH,
    'resume_mode': RESUME,
    'resumed_this_session': bool(RESUME_FROM),
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': GPU_NAME,
    'seed': EXPECTED['seed'],
    'augmentation': fingerprint['augmentation_spec'],
    'loss': fingerprint['loss_spec'],
    'epochs_recorded': len(final_history),
    'best_checkpoint_epoch': best_info['epoch'],
    'validation_cases': list(split.val_ids),
    'test_split_accessed': False,
    'external_validation_accessed': False,
}
repro_path = os.path.join(E2_DRIVE_DIR, 'reproducibility.json')
with open(repro_path, 'w', encoding='utf-8') as f:
    json.dump(repro, f, indent=2, default=str)
print('Reproducibility record:', repro_path)
print(json.dumps(repro, indent=2, default=str)[:1500])


## 23. Final experiment summary


In [ ]:
import json

with open(os.path.join(E2_DRIVE_DIR, 'summary.json'), encoding='utf-8') as f:
    e2_summary = json.load(f)
macro = e2_summary['macro_dice']
aug = e2_summary.get('augmentation') or {}

print('=' * 78)
print('EXPERIMENT E2 -- FINAL SUMMARY')
print('=' * 78)
print(f"  Experiment ID           : {e2_summary['experiment']}")
print(f"  Design cell             : CE loss x augmentation ON")
print(f"  Git commit              : {e2_summary.get('git_commit')}")
print( '  Dataset                 : Prostate158 (official train archive)')
print( '  Target                  : t2_anatomy_reader1 (0=bg, 1=CG, 2=PZ)')
print( '  Split                   : 119 train / 20 validation (official)')
print( '  Architecture            : 2D U-Net (unchanged from baseline)')
print( '  Loss                    : Cross Entropy (unchanged from baseline)')
print( '  Optimizer / LR / WD     : AdamW / 1e-3 / 1e-4')
print( '  Batch size              : 8')
print( '  Seed                    : 42')
print( '  Precision / AMP         : FP32 / OFF')
print( '  Scheduler               : NONE')
print('-' * 78)
print( '  AUGMENTATION (the E2 factor, training slices only)')
print(f"    rotation              : +/-{aug.get('rotation_degrees')} deg, p={aug.get('rotation_p')}")
print(f"    scale                 : {aug.get('scale_min')}-{aug.get('scale_max')}, p={aug.get('scale_p')}")
print(f"    horizontal flip       : p={aug.get('hflip_p')}")
print(f"    gamma (image only)    : {aug.get('gamma_min')}-{aug.get('gamma_max')}, p={aug.get('gamma_p')}")
print(f"    image / mask interp   : bilinear / nearest-neighbour")
print(f"    augmentation seed     : {aug.get('seed')}")
print( '    validation augmented  : NO')
print('-' * 78)
print( '  Maximum epochs          : 100')
print(f"  Epochs completed        : {e2_summary.get('epochs_completed')}")
print(f"  Epoch records on disk   : {e2_summary.get('epoch_records_on_disk')}")
print(f"  Best checkpoint epoch   : {e2_summary.get('best_checkpoint_epoch')}")
print(f"  Best val Dice proxy     : {e2_summary.get('best_val_dice_proxy')}")
print(f"  Resumed this session    : {bool(RESUME_FROM)}")
print('-' * 78)
for cid, label in (('1', 'CG'), ('2', 'PZ')):
    cell = e2_summary['per_class'][cid]['dice']
    print(f"  {label} Dice   baseline {cell['baseline']:.4f} | E1 "
          f"{cell['e1'] if cell['e1'] is None else format(cell['e1'], '.4f')} | "
          f"E2 {cell['e2']:.4f}  ({cell['delta_vs_baseline']:+.4f} vs baseline)")
print(f"  MACRO Dice baseline {macro['baseline']:.4f} | E1 "
      f"{macro['e1'] if macro['e1'] is None else format(macro['e1'], '.4f')} | "
      f"E2 {macro['e2']:.4f}  ({macro['delta_vs_baseline']:+.4f} vs baseline)")
print('-' * 78)
print(f"  VERDICT: E2 {e2_summary['verdict'].upper()} vs baseline (validation macro Dice)")
print('-' * 78)
print( '  Validation cases        : 20')
print( '  Official test cases used: 0')
print( '  External validation     : none')
print('=' * 78)
print()
print('Artifacts on Drive:', E2_DRIVE_DIR)
for entry in sorted(os.listdir(E2_DRIVE_DIR)):
    full = os.path.join(E2_DRIVE_DIR, entry)
    marker = '/' if os.path.isdir(full) else ''
    print(f'    {entry}{marker}')
print()
print('The official 19-case test set was NOT used during E2 training, model')
print('selection, or experiment comparison. No external validation was accessed.')
print()
print('NEXT STEPS (not done here):')
print('  1. Report E2 alongside the baseline and E1 to the supervisor.')
print('  2. Train the fourth cell of the 2x2 design, E1+E2 (Dice+CE + augmentation).')
print('  3. Only after that: the factorial analysis, then final model selection,')
print('     and only then the single official test evaluation.')
